In [173]:
import torch
from torch import nn
import torch.nn.functional as F

## Hyper parameters
max_epoch = 5000
learning_rate = 1e-3
eval_interval = 100

batch_size = 4
block_size = 8

torch.manual_seed(24)


# 1 Load the data
with open("shakes.txt", "r") as f:
    text = f.read()
    
# 2 preporcess the data
## create preprocesor - tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)

## int : str mapping
itos = {i:st for i,st in enumerate(chars)}
stoi = {st:i for i,st in enumerate(chars)}

# encode decode
encode = lambda inp : [stoi[i] for i in inp]
decode = lambda inp : "".join([itos[i] for i in inp])

data = torch.tensor(encode(text))


In [174]:
## split the data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return x, y

xb, yb = get_batch("train")


In [175]:
### 3. create model
class BiagramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        ## This will convetr (B,T) into (B,T,65(vocab_size))--> B,T,C
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) ## 
        
    def forward(self, inpx, target= None):
        
        logits = self.token_embedding_table(inpx) ## B,T ---> BTC
        B,T,C = logits.shape
        if target is None:
            loss = None
        else:
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
            
        return logits, loss
    
    def generate(self, idx, max_length):
        for i in range(max_length):
            logit, _= self(idx)
            logit = logit[:,-1,:]
            probs =  torch.softmax(logit, dim=-1)
            print(probs)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx,idx_next), dim=1)
        return idx
        
            
        
    

model = BiagramModel(vocab_size=vocab_size)
logit, loss = model(xb,yb)


        

In [176]:
### Training 
train_losses = []
val_losses = []
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for epoch in range(max_epoch):
    running_loss = 0.0
    model.train()
    xb, yb = get_batch("train")
    _, loss = model(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    


    
    if epoch % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            x,y = get_batch("val")
            _, val_loss = model(x,y)
            print(f"The Epoch : {epoch}  | Train loss  :{loss.item()} | Val Loss: {val_loss}")

            
        
   
   
    
    


The Epoch : 0  | Train loss  :4.5098676681518555 | Val Loss: 4.51715612411499
The Epoch : 100  | Train loss  :4.710963726043701 | Val Loss: 4.505226135253906
The Epoch : 200  | Train loss  :4.511392116546631 | Val Loss: 4.51486349105835
The Epoch : 300  | Train loss  :4.856550693511963 | Val Loss: 4.734537601470947
The Epoch : 400  | Train loss  :4.423707485198975 | Val Loss: 4.072881698608398
The Epoch : 500  | Train loss  :4.7307000160217285 | Val Loss: 4.328149318695068
The Epoch : 600  | Train loss  :4.390899181365967 | Val Loss: 4.343342304229736
The Epoch : 700  | Train loss  :4.051155090332031 | Val Loss: 3.912344217300415
The Epoch : 800  | Train loss  :4.0766987800598145 | Val Loss: 4.127737998962402
The Epoch : 900  | Train loss  :3.937652826309204 | Val Loss: 4.296674728393555
The Epoch : 1000  | Train loss  :4.068939685821533 | Val Loss: 4.149411678314209
The Epoch : 1100  | Train loss  :4.108245849609375 | Val Loss: 3.9530205726623535
The Epoch : 1200  | Train loss  :3.832

In [177]:
torch.manual_seed(42)
a = torch.ones(3,3)
b =  torch.randint(10, size=(3,2), dtype=torch.float32)
a, b

(tensor([[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]),
 tensor([[2., 7.],
         [6., 4.],
         [6., 5.]]))

In [178]:
c = a@b
c

tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])

In [179]:
z = torch.tril(torch.ones(3,3))
z

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [180]:
z.sum(1, keepdim=True)

tensor([[1.],
        [2.],
        [3.]])

In [181]:
tril  = torch.tril(torch.ones(3,3))
wei = torch.zeros(3,3)

tril, wei

(tensor([[1., 0., 0.],
         [1., 1., 0.],
         [1., 1., 1.]]),
 tensor([[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]))

In [182]:
wei = wei.masked_fill(tril == 0 , float("-inf"))
wei

tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])

In [183]:
wei.softmax(1)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [188]:
torch.manual_seed(42)
B,T,C = 4,8,32
x = torch.rand(B,T,C)
head_size = 16

query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x) # B,T, 16
k = key(x)
v = value(x)
wei = q @ k.transpose(-2,-1)  ## B,T,T

tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros(T,T)
wei =  wei.masked_fill(tril == 0, float("-inf"))
wei = wei.softmax(-1)
# out =  wei @ x
out = wei@v
out.shape

torch.Size([4, 8, 16])

In [189]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4820, 0.5180, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3349, 0.3342, 0.3309, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2726, 0.2886, 0.2287, 0.2101, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2131, 0.2323, 0.2080, 0.1549, 0.1916, 0.0000, 0.0000, 0.0000],
         [0.1638, 0.1688, 0.1754, 0.1456, 0.1855, 0.1610, 0.0000, 0.0000],
         [0.1484, 0.1446, 0.1501, 0.1120, 0.1332, 0.1373, 0.1743, 0.0000],
         [0.1268, 0.1322, 0.1378, 0.1287, 0.1132, 0.1154, 0.1157, 0.1303]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4987, 0.5013, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3456, 0.3618, 0.2926, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2693, 0.2620, 0.2218, 0.2468, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1897, 0.1899, 0.1825, 0.2116, 0.2264, 0.0000, 0.0000, 0.0000],
         [0.1620, 0.163